In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader

# Detectar Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Executando no Google Colab")
except:
    IN_COLAB = False
    print("✓ Executando localmente")

# Setup Path
if IN_COLAB:
    if not os.path.exists('/content/ufc-easytpp'):
        print("Clonando repositório...")
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
else:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- CORREÇÃO CRÍTICA PARA AMP (FP16) ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
import math
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        # CORREÇÃO: -1e4 em vez de -1e9 para evitar NaN em FP16
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

baselayer.attention = attention_fixed

# Patch nos módulos que importaram 'attention' diretamente
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except ImportError: pass
try:
    import easy_tpp.model.torch_model.torch_rothp_hybrid
    easy_tpp.model.torch_model.torch_rothp_hybrid.attention = attention_fixed
except ImportError: pass

print("✓ Patch FP16 aplicado")
# --------------------------------------

# Importar Modelos
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_thp_expdecay import THPExpDecay
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_rothp_hybrid import RoTHPHybrid
from easy_tpp.model.torch_model.torch_smurf import SmurfTHP

import importlib
import easy_tpp.model.torch_model.torch_smurf
importlib.reload(easy_tpp.model.torch_model.torch_smurf)
from easy_tpp.model.torch_model.torch_smurf import SmurfTHP
print("✓ Módulo SMURF recarregado")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Desabilitar CuDNN Benchmark para evitar erros de execução em segunda derivada
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("✓ CuDNN Benchmark desabilitado para estabilidade do SMURF")


In [ ]:
print("Carregando dataset 'easytpp/retweet'...")
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']
all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])
TIME_SCALE = np.mean(all_deltas)
print(f"Time Scale: {TIME_SCALE:.4f}")
NUM_EVENT_TYPES = 3
PAD_TOKEN_ID = NUM_EVENT_TYPES
NUM_EVENT_TYPES_PAD = NUM_EVENT_TYPES + 1


In [ ]:
def collate_fn_opt(batch_list):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), PAD_TOKEN_ID, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / TIME_SCALE
        td = td / TIME_SCALE
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0
class ModelConfig:
    def __init__(self):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = NUM_EVENT_TYPES
        self.num_event_types_pad = NUM_EVENT_TYPES_PAD
        self.pad_token_id = PAD_TOKEN_ID
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()
config = ModelConfig()


In [ ]:
def compute_metrics_full(model, dataset_or_loader, eval_device=None):
    if eval_device is None:
        eval_device = next(model.parameters()).device
        
    if isinstance(dataset_or_loader, DataLoader):
        dataset = dataset_or_loader.dataset
    else:
        dataset = dataset_or_loader
        
    eval_batch_size = 256 
    loader = DataLoader(dataset, batch_size=eval_batch_size, shuffle=False, collate_fn=collate_fn_opt, num_workers=0)
    
    model.eval()
    total_acc = 0
    total_rmse = 0
    total_nll = 0
    total_events = 0
    with torch.no_grad():
        for batch in loader:
            batch_gpu = tuple(t.to(eval_device) for t in batch)
            
            with torch.enable_grad():
                 loss, num_events = model.loglike_loss(batch_gpu)
            total_nll += loss.item()
            
            _, time_delta_target, type_target, mask_target, _ = batch_gpu
            
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch_gpu)
            
            target_types = type_target[:, 1:]
            target_deltas = time_delta_target[:, 1:]
            target_mask = mask_target[:, 1:]
            
            correct = (types_pred == target_types) * target_mask
            total_acc += correct.sum().item()
            se = ((dtimes_pred - target_deltas) ** 2) * target_mask
            total_rmse += se.sum().item()
            total_events += target_mask.sum().item()
    avg_nll = total_nll / (total_events + 1e-9)
    avg_acc = total_acc / (total_events + 1e-9)
    avg_rmse = np.sqrt(total_rmse / (total_events + 1e-9))
    return avg_nll, avg_acc, avg_rmse


In [ ]:
BATCH_SIZE = 2048
EPOCHS = 20 
LR = 1e-3
NUM_WORKERS = 0

train_loader_default = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_opt, num_workers=NUM_WORKERS, pin_memory=True)
train_loader_small = DataLoader(train_data, batch_size=128, shuffle=True, collate_fn=collate_fn_opt, num_workers=NUM_WORKERS, pin_memory=True)
dev_loader = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate_fn_opt, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate_fn_opt, num_workers=NUM_WORKERS, pin_memory=True)

results = {}
final_test_results = {}

def train_model(model_class, model_name):
    print(f"\n{'='*40}")
    print(f"Treinando: {model_name}")
    print(f"{'='*40}")
    
    # Dispositivo de Treinamento
    # Forçar CPU para SMURF para evitar erro de CUBLAS em derivadas segundas
    if model_name == "SMURF-THP":
        train_device = torch.device('cpu')
        print("-> Usando CPU para SMURF (Workaround para estabilidade)")
        current_train_loader = train_loader_small
        use_amp = False
    else:
        train_device = device # GPU
        current_train_loader = train_loader_default
        use_amp = True
        
    model = model_class(config).to(train_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda') if use_amp and train_device.type == 'cuda' else None
    
    history = {'train_nll': [], 'val_nll': []}
    start_time = time.time()
    best_val_nll = float('inf')
    checkpoint_path = f"best_model_{model_name.replace(' ', '_')}.pth"
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        total_events = 0
        for batch in current_train_loader:
            batch = [t.to(train_device, non_blocking=True) for t in batch]
            optimizer.zero_grad(set_to_none=True)
            
            if use_amp:
                with torch.amp.autocast('cuda'):
                    loss, num_events = model.loglike_loss(batch)
                    loss_norm = loss / (num_events + 1e-9)
                scaler.scale(loss_norm).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss, num_events = model.loglike_loss(batch)
                loss_norm = loss / (num_events + 1e-9)
                loss_norm.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            total_events += num_events
        
        train_nll = total_loss / (total_events + 1e-9)
        
        model.eval()
        val_loss = 0
        val_events = 0
        with torch.enable_grad():
            for batch in dev_loader:
                batch = [t.to(train_device, non_blocking=True) for t in batch]
                loss, num_events = model.loglike_loss(batch)
                val_loss += loss.item()
                val_events += num_events
        val_nll = val_loss / (val_events + 1e-9)
        
        history['train_nll'].append(train_nll)
        history['val_nll'].append(val_nll)
        print(f"Ep {epoch:02d} | Train Loss: {train_nll:.4f} | Val Loss: {val_nll:.4f}")
        
        if val_nll < best_val_nll:
            best_val_nll = val_nll
            torch.save(model.state_dict(), checkpoint_path)

    print(f">>> Treino finalizado em {time.time() - start_time:.1f}s")
    model.load_state_dict(torch.load(checkpoint_path))
    print(f"Calculando métricas finais no TESTE (Langevin/Thinning)...")
    test_nll, test_acc, test_rmse = compute_metrics_full(model, test_data, eval_device=train_device)
    print(f">>> RESULTADO FINAL (TESTE) - {model_name}:")
    print(f"    Loss (SM): {test_nll:.4f}")
    print(f"    Acc:       {test_acc:.4f}")
    print(f"    RMSE:      {test_rmse:.4f}")
    final_test_results[model_name] = {'nll': test_nll, 'acc': test_acc, 'rmse': test_rmse}
    return model, history

# Comparar THP Decay vs SMURF-THP
models_to_run = [
    (THPExpDecay, "THP Decay"),
    (SmurfTHP, "SMURF-THP")
]

trained_models = {}
for cls, name in models_to_run:
    model, hist = train_model(cls, name)
    results[name] = hist
    trained_models[name] = model

# Tabela
import pandas as pd
print("\n=== TABELA FINAL ===")
print(pd.DataFrame(final_test_results).T)
